# AIC 2026 — Object detection ingestion · WeDetect Large

**Input:** `aqpahm/aic2026-keyframes-transnetv2`  
**Output:** `aqpahm/aic2026-od-wedetect-large`  
**Artifacts:** `detections.parquet`, `frames.parquet`, `_OD_SUCCESS.json`

Notebook nguồn chạy độc lập trên **Google Colab hoặc Kaggle**. Threshold hiện tại
là 0.30; đây là tham số của artifact OD, chưa phải quyết định về retrieval.


In [ ]:
%pip install -q -U "huggingface_hub>=0.34,<2" "transformers==4.57.1" "mmengine>=0.10,<1" pyarrow pycocotools



In [ ]:
import os
import sys
import tempfile

# Các biến này phải được đặt trước lần import huggingface_hub đầu tiên.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"

import gc
import hashlib
import importlib.util
import json
import queue
import re
import shutil
import subprocess
import sys
import tarfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import pandas as pd
import torch
from PIL import Image
from PIL import ImageDraw
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download

INPUT_REPO = "aqpahm/aic2026-keyframes-transnetv2"
MODEL_VARIANT = "large"  # đổi thành "base" nếu Large không vừa T4
OUTPUT_REPO = f"aqpahm/aic2026-od-wedetect-large"
MODEL_REPO = "fushh7/WeDetect"
MODEL_FILENAME = f"wedetect_{MODEL_VARIANT}.pth"
MODEL_ID = f"WeChatCV/WeDetect-{MODEL_VARIANT}"

IMAGE_SIZE = 1280 if MODEL_VARIANT == "large" else 640
CONFIDENCE_THRESHOLD = 0.30
IOU_THRESHOLD = 0.70
MAX_DETECTIONS = 150
INITIAL_BATCH_SIZE = 2  # OOM recovery halves this automatically
# Colab dễ ngắt phiên hơn Kaggle; commit mỗi 5 video để giảm phần phải chạy lại.
UPLOAD_BATCH_VIDEOS = 10
# Tiếp tục toàn bộ video còn thiếu; các video hoàn chỉnh trên HF được tự bỏ qua.
MAX_VIDEOS_PER_RUN = None
QA_PREVIEW_COUNT = 10

def detect_runtime():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if RUNTIME == "colab":
        from google.colab import userdata
        value = userdata.get(name)
    elif RUNTIME == "kaggle":
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    if not value:
        raise RuntimeError(
            f"Missing {name}. Add it as a Colab/Kaggle secret and enable notebook access."
        )
    return value


def hosted_work_root(job_name):
    if RUNTIME == "colab":
        return Path("/content") / job_name
    if RUNTIME == "kaggle":
        return Path("/kaggle/temp") / job_name
    return Path(tempfile.gettempdir()) / job_name

RUNTIME = detect_runtime()
ROOT = hosted_work_root("aic-od-wedetect")
DOWNLOAD_ROOT = ROOT / "downloads"
OUTPUT_ROOT = ROOT / "output"
DECODE_ROOT = ROOT / "decoded"
PREVIEW_ROOT = ROOT / "preview"
SOURCE_ROOT = ROOT / "WeDetect"
for directory in (DOWNLOAD_ROOT, OUTPUT_ROOT, DECODE_ROOT, PREVIEW_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

HF_TOKEN = read_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)
account = api.whoami()
print("Runtime:", RUNTIME)
print("Hugging Face:", account["name"])

api.create_repo(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    private=True,
    exist_ok=True,
)
INPUT_REVISION = api.dataset_info(INPUT_REPO, token=HF_TOKEN).sha
print("Input revision:", INPUT_REVISION)
print("Output:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in the Colab/Kaggle runtime before running this job")

GPU_IDS = list(range(torch.cuda.device_count()))
CPU_THREADS_PER_WORKER = max(1, min(4, (os.cpu_count() or 2) // len(GPU_IDS)))
torch.set_num_threads(CPU_THREADS_PER_WORKER)
print("GPU workers:", len(GPU_IDS))
for device_id in GPU_IDS:
    properties = torch.cuda.get_device_properties(device_id)
    print(f"  cuda:{device_id}: {properties.name}, {properties.total_memory / 1024**3:.1f} GiB")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

torch.backends.cudnn.benchmark = True



In [ ]:
# Core vocab cố định được kiểm tra trên 210 query AIC 2025–2026.
# WeDetect infer bằng prompt zh; en/vi được lưu để search và hiển thị.
CORE_VOCAB_VERSION = "aic-query-2025-2026-v1-400"
LABEL_SPECS = [
    ('person', 'người', '人'),
    ('bicycle', 'xe đạp', '自行车'),
    ('car', 'ô tô', '汽车'),
    ('motorcycle', 'xe máy', '摩托车'),
    ('airplane', 'máy bay', '飞机'),
    ('bus', 'xe buýt', '公共汽车'),
    ('train', 'tàu hỏa', '火车'),
    ('truck', 'xe tải', '卡车'),
    ('boat', 'thuyền', '船'),
    ('traffic light', 'đèn giao thông', '交通灯'),
    ('fire hydrant', 'trụ cứu hỏa', '消防栓'),
    ('stop sign', 'biển dừng', '停车标志'),
    ('bench', 'ghế dài', '长凳'),
    ('bird', 'chim', '鸟'),
    ('cat', 'mèo', '猫'),
    ('dog', 'chó', '狗'),
    ('horse', 'ngựa', '马'),
    ('sheep', 'cừu', '羊'),
    ('cow', 'bò', '牛'),
    ('elephant', 'voi', '大象'),
    ('bear', 'gấu', '熊'),
    ('zebra', 'ngựa vằn', '斑马'),
    ('giraffe', 'hươu cao cổ', '长颈鹿'),
    ('backpack', 'ba lô', '背包'),
    ('umbrella', 'ô', '雨伞'),
    ('handbag', 'túi xách', '手提包'),
    ('tie', 'cà vạt', '领带'),
    ('suitcase', 'va li', '手提箱'),
    ('sports ball', 'quả bóng', '运动球'),
    ('kite', 'diều', '风筝'),
    ('baseball bat', 'gậy bóng chày', '棒球棒'),
    ('baseball glove', 'găng bóng chày', '棒球手套'),
    ('skateboard', 'ván trượt', '滑板'),
    ('surfboard', 'ván lướt', '冲浪板'),
    ('tennis racket', 'vợt tennis', '网球拍'),
    ('bottle', 'chai', '瓶子'),
    ('wine glass', 'ly rượu', '酒杯'),
    ('cup', 'cốc', '杯子'),
    ('fork', 'nĩa', '叉子'),
    ('knife', 'dao', '刀'),
    ('spoon', 'thìa', '勺子'),
    ('bowl', 'bát', '碗'),
    ('banana', 'chuối', '香蕉'),
    ('apple', 'táo', '苹果'),
    ('sandwich', 'bánh mì kẹp', '三明治'),
    ('orange', 'cam', '橙子'),
    ('broccoli', 'bông cải', '西兰花'),
    ('carrot', 'cà rốt', '胡萝卜'),
    ('hot dog', 'xúc xích', '热狗'),
    ('pizza', 'pizza', '披萨'),
    ('donut', 'bánh vòng', '甜甜圈'),
    ('cake', 'bánh', '蛋糕'),
    ('chair', 'ghế', '椅子'),
    ('couch', 'ghế sofa', '沙发'),
    ('potted plant', 'cây cảnh', '盆栽植物'),
    ('bed', 'giường', '床'),
    ('dining table', 'bàn ăn', '餐桌'),
    ('toilet', 'bồn cầu', '厕所'),
    ('television', 'tivi', '电视显示器'),
    ('laptop', 'laptop', '笔记本电脑'),
    ('mouse', 'chuột máy tính', '鼠标'),
    ('remote control', 'điều khiển', '遥控器'),
    ('keyboard', 'bàn phím', '键盘'),
    ('cell phone', 'điện thoại', '手机'),
    ('microwave', 'lò vi sóng', '微波炉'),
    ('oven', 'lò nướng', '烤箱'),
    ('toaster', 'máy nướng bánh', '烤面包机'),
    ('sink', 'bồn rửa', '水槽'),
    ('refrigerator', 'tủ lạnh', '冰箱'),
    ('book', 'sách', '书'),
    ('clock', 'đồng hồ', '时钟'),
    ('vase', 'bình hoa', '花瓶'),
    ('scissors', 'kéo', '剪刀'),
    ('teddy bear', 'gấu bông', '泰迪熊'),
    ('hair dryer', 'máy sấy tóc', '吹风机'),
    ('toothbrush', 'bàn chải', '牙刷'),
    ('helmet', 'mũ bảo hiểm', '头盔'),
    ('microphone', 'micro', '麦克风'),
    ('flag', 'lá cờ', '旗帜'),
    ('billboard', 'bảng quảng cáo', '广告牌'),
    ('signboard', 'biển hiệu', '招牌'),
    ('storefront', 'mặt tiền cửa hàng', '店面'),
    ('building', 'tòa nhà', '建筑物'),
    ('bridge', 'cầu', '桥'),
    ('tower', 'tháp', '塔'),
    ('temple', 'đền', '寺庙'),
    ('church', 'nhà thờ', '教堂'),
    ('pagoda', 'chùa', '宝塔'),
    ('monument', 'đài tưởng niệm', '纪念碑'),
    ('statue', 'tượng', '雕像'),
    ('traffic cone', 'cọc giao thông', '交通锥'),
    ('license plate', 'biển số xe', '车牌'),
    ('excavator', 'máy xúc', '挖掘机'),
    ('bulldozer', 'xe ủi', '推土机'),
    ('crane', 'cần cẩu', '起重机'),
    ('factory', 'nhà máy', '工厂'),
    ('rice field', 'ruộng lúa', '稻田'),
    ('river', 'sông', '河流'),
    ('beach', 'bãi biển', '海滩'),
    ('mountain', 'núi', '山'),
    ('waterfall', 'thác nước', '瀑布'),
    ('conical hat', 'nón lá', '斗笠'),
    ('traditional dress', 'áo dài', '越南传统服装'),
    ('lion dance costume', 'đầu lân', '舞狮服装'),
    ('cooking pot', 'nồi nấu', '锅'),
    ('frying pan', 'chảo', '煎锅'),
    ('rice cooker', 'nồi cơm điện', '电饭锅'),
    ('chopsticks', 'đũa', '筷子'),
    ('fishing net', 'lưới đánh cá', '渔网'),
    ('wooden boat', 'ghe gỗ', '木船'),
    ('camera', 'máy ảnh', '相机'),
    ('video camera', 'máy quay', '摄像机'),
    ('news desk', 'bàn dẫn chương trình', '新闻主播台'),
    ('astronaut', 'phi hành gia', '宇航员'),
    ('cardboard', 'bìa các-tông', '纸板'),
    ('portrait', 'ảnh chân dung', '人物肖像'),
    ('chocolate', 'sô-cô-la', '巧克力'),
    ('cotton swab', 'tăm bông', '棉签'),
    ('pillar', 'cột', '柱子'),
    ('dragon', 'rồng', '龙'),
    ('certificate', 'giấy chứng nhận', '证书'),
    ('fire', 'lửa', '火焰'),
    ('mint', 'bạc hà', '薄荷叶'),
    ('robot', 'rô-bốt', '机器人'),
    ('robot arm', 'cánh tay rô-bốt', '机械臂'),
    ('cyclist', 'người đi xe đạp', '骑自行车的人'),
    ('fence', 'hàng rào', '栅栏'),
    ('durian', 'sầu riêng', '榴莲'),
    ('water buffalo', 'trâu nước', '水牛'),
    ('fish trap', 'lờ bắt cá', '捕鱼笼'),
    ('wicker basket', 'giỏ đan', '藤编篮'),
    ('school uniform', 'đồng phục học sinh', '校服'),
    ('graduation gown', 'áo tốt nghiệp', '学位服'),
    ('car chassis', 'khung gầm ô tô', '汽车底盘'),
    ('handpan', 'đàn handpan', '手碟'),
    ('market stall', 'sạp chợ', '市场摊位'),
    ('bamboo basket', 'giỏ tre', '竹篮'),
    ('mangosteen', 'măng cụt', '山竹'),
    ('pomelo', 'bưởi', '柚子'),
    ('longan', 'nhãn', '龙眼'),
    ('dragon dance costume', 'trang phục múa rồng', '舞龙服装'),
    ('paper bag', 'túi giấy', '纸袋'),
    ('gong', 'cồng chiêng', '锣'),
    ('space shuttle', 'tàu con thoi', '航天飞机'),
    ('baseball cap', 'mũ lưỡi trai', '棒球帽'),
    ('tray', 'khay', '托盘'),
    ('strawberry', 'dâu tây', '草莓'),
    ('camera lens', 'ống kính máy ảnh', '相机镜头'),
    ('sculpture', 'tác phẩm điêu khắc', '雕塑'),
    ('roller skate', 'giày patin', '旱冰鞋'),
    ('mushroom', 'nấm', '蘑菇'),
    ('bean curd', 'đậu phụ', '豆腐'),
    ('stove', 'bếp', '炉灶'),
    ('tiger', 'hổ', '老虎'),
    ('grape', 'nho', '葡萄'),
    ('shark', 'cá mập', '鲨鱼'),
    ('goat', 'dê', '山羊'),
    ('pineapple', 'dứa', '菠萝'),
    ('octopus', 'bạch tuộc', '章鱼'),
    ('squid', 'mực', '鱿鱼'),
    ('barrow', 'xe cút kít', '手推车'),
    ('turtle', 'rùa', '龟'),
    ('fan', 'quạt', '风扇'),
    ('poster', 'áp phích', '海报'),
    ('map', 'bản đồ', '地图'),
    ('blackboard', 'bảng đen', '黑板'),
    ('projector', 'máy chiếu', '投影仪'),
    ('notebook', 'vở', '笔记本'),
    ('life jacket', 'áo phao', '救生衣'),
    ('raincoat', 'áo mưa', '雨衣'),
    ('drum', 'trống', '鼓'),
    ('battery', 'pin', '电池'),
    ('motor scooter', 'xe tay ga', '小型摩托车'),
    ('pickup truck', 'xe bán tải', '皮卡车'),
    ('cab', 'taxi', '出租车'),
    ('minivan', 'xe đa dụng', '小型货车'),
    ('trailer truck', 'xe đầu kéo', '牵引式挂车'),
    ('garbage truck', 'xe rác', '垃圾车'),
    ('fire engine', 'xe cứu hỏa', '消防车'),
    ('ambulance', 'xe cứu thương', '救护车'),
    ('school bus', 'xe buýt trường học', '校车'),
    ('helicopter', 'trực thăng', '直升机'),
    ('hot-air balloon', 'khinh khí cầu', '热气球'),
    ('canoe', 'xuồng', '独木舟'),
    ('kayak', 'thuyền kayak', '皮划艇'),
    ('ferry', 'phà', '渡船'),
    ('yacht', 'du thuyền', '游艇'),
    ('barge', 'sà lan', '驳船'),
    ('tractor', 'máy kéo', '拖拉机'),
    ('forklift', 'xe nâng', '叉车'),
    ('wheelchair', 'xe lăn', '轮椅'),
    ('baby buggy', 'xe đẩy em bé', '婴儿车'),
    ('street sign', 'biển tên đường', '路标'),
    ('parking meter', 'đồng hồ đỗ xe', '停车计时器'),
    ('monkey', 'khỉ', '猴子'),
    ('gorilla', 'khỉ đột', '猩猩'),
    ('deer', 'hươu', '鹿'),
    ('camel', 'lạc đà', '骆驼'),
    ('hog', 'lợn', '猪'),
    ('rhinoceros', 'tê giác', '犀牛'),
    ('hippopotamus', 'hà mã', '河马'),
    ('lion', 'sư tử', '狮子'),
    ('wolf', 'sói', '狼'),
    ('rabbit', 'thỏ', '兔子'),
    ('squirrel', 'sóc', '松鼠'),
    ('rat', 'chuột cống', '老鼠'),
    ('dolphin', 'cá heo', '海豚'),
    ('fish', 'cá', '鱼'),
    ('goldfish', 'cá vàng', '金鱼'),
    ('crab', 'cua', '螃蟹'),
    ('prawn', 'tôm', '虾'),
    ('frog', 'ếch', '青蛙'),
    ('snake', 'rắn', '蛇'),
    ('lizard', 'thằn lằn', '蜥蜴'),
    ('chicken', 'gà', '鸡'),
    ('cock', 'gà trống', '公鸡'),
    ('duck', 'vịt', '鸭子'),
    ('goose', 'ngỗng', '鹅'),
    ('parrot', 'vẹt', '鹦鹉'),
    ('pigeon', 'bồ câu', '鸽子'),
    ('eagle', 'đại bàng', '鹰'),
    ('owl', 'cú', '猫头鹰'),
    ('butterfly', 'bướm', '蝴蝶'),
    ('beetle', 'bọ cánh cứng', '甲虫'),
    ('spider', 'nhện', '蜘蛛'),
    ('watermelon', 'dưa hấu', '西瓜'),
    ('melon', 'dưa', '瓜'),
    ('coconut', 'dừa', '椰子'),
    ('papaya', 'đu đủ', '木瓜'),
    ('pear', 'lê', '梨'),
    ('peach', 'đào', '桃子'),
    ('lemon', 'chanh vàng', '柠檬'),
    ('lime', 'chanh xanh', '酸橙'),
    ('avocado', 'bơ', '鳄梨'),
    ('kiwi fruit', 'kiwi', '猕猴桃'),
    ('cherry', 'anh đào', '樱桃'),
    ('blueberry', 'việt quất', '蓝莓'),
    ('raspberry', 'mâm xôi', '树莓'),
    ('fig', 'sung', '无花果'),
    ('edible corn', 'ngô', '可食用玉米'),
    ('potato', 'khoai tây', '土豆'),
    ('sweet potato', 'khoai lang', '红薯'),
    ('tomato', 'cà chua', '番茄'),
    ('cucumber', 'dưa leo', '黄瓜'),
    ('pumpkin', 'bí đỏ', '南瓜'),
    ('eggplant', 'cà tím', '茄子'),
    ('onion', 'hành tây', '洋葱'),
    ('garlic', 'tỏi', '大蒜'),
    ('ginger', 'gừng', '姜'),
    ('chili', 'ớt', '辣椒'),
    ('bell pepper', 'ớt chuông', '甜椒'),
    ('cauliflower', 'súp lơ trắng', '花椰菜'),
    ('lettuce', 'xà lách', '生菜'),
    ('pea', 'đậu Hà Lan', '豌豆'),
    ('beef', 'thịt bò', '牛肉'),
    ('sausage', 'xúc xích', '香肠'),
    ('egg', 'trứng', '蛋'),
    ('bread', 'bánh mì', '面包'),
    ('soup', 'súp', '汤'),
    ('salad', 'xa lát', '沙拉'),
    ('cookie', 'bánh quy', '饼干'),
    ('cracker', 'bánh giòn', '薄脆饼干'),
    ('pancake', 'bánh kếp', '薄煎饼'),
    ('water bottle', 'chai nước', '水瓶'),
    ('beer bottle', 'chai bia', '啤酒瓶'),
    ('wine bottle', 'chai rượu', '葡萄酒瓶'),
    ('teapot', 'ấm trà', '茶壶'),
    ('kettle', 'ấm đun nước', '水壶'),
    ('plate', 'đĩa', '盘子'),
    ('saucer', 'đĩa lót', '茶碟'),
    ('jar', 'lọ', '罐子'),
    ('can', 'lon', '罐头'),
    ('chopping board', 'thớt', '案板'),
    ('spatula', 'xẻng lật thức ăn', '抹刀'),
    ('ladle', 'muôi', '长柄勺'),
    ('tongs', 'kẹp gắp', '钳子'),
    ('peeler', 'dao bào', '削皮器'),
    ('grater', 'dụng cụ bào', '擦菜板'),
    ('colander', 'rổ lọc', '滤器'),
    ('blender', 'máy xay sinh tố', '搅拌机'),
    ('food processor', 'máy xay thực phẩm', '食品加工机'),
    ('coffee maker', 'máy pha cà phê', '咖啡机'),
    ('dishwasher', 'máy rửa bát', '洗碗机'),
    ('apron', 'tạp dề', '围裙'),
    ('shirt', 'áo sơ mi', '衬衫'),
    ('sweater', 'áo len', '毛衣'),
    ('jacket', 'áo khoác', '夹克衫'),
    ('coat', 'áo choàng', '外套'),
    ('suit', 'bộ vest', '套装'),
    ('dress', 'váy liền', '连衣裙'),
    ('skirt', 'chân váy', '裙子'),
    ('trousers', 'quần dài', '裤子'),
    ('short pants', 'quần ngắn', '短裤'),
    ('vest', 'áo ghi lê', '背心'),
    ('swimsuit', 'đồ bơi', '泳衣'),
    ('pajamas', 'đồ ngủ', '睡衣'),
    ('sock', 'tất', '短袜'),
    ('shoe', 'giày', '鞋'),
    ('sandal', 'dép quai', '凉鞋'),
    ('slipper', 'dép lê', '拖鞋'),
    ('boot', 'ủng', '靴子'),
    ('hat', 'mũ', '帽子'),
    ('glove', 'găng tay', '手套'),
    ('scarf', 'khăn quàng', '围巾'),
    ('belt', 'thắt lưng', '皮带'),
    ('sunglasses', 'kính râm', '太阳镜'),
    ('spectacles', 'kính mắt', '眼镜'),
    ('watch', 'đồng hồ đeo tay', '手表'),
    ('necklace', 'vòng cổ', '项链'),
    ('earring', 'bông tai', '耳环'),
    ('bracelet', 'vòng tay', '手镯'),
    ('ring', 'nhẫn', '戒指'),
    ('wallet', 'ví', '钱包'),
    ('shopping bag', 'túi mua sắm', '购物袋'),
    ('briefcase', 'cặp tài liệu', '公文包'),
    ('desk', 'bàn làm việc', '书桌'),
    ('stool', 'ghế đẩu', '凳子'),
    ('armchair', 'ghế bành', '扶手椅'),
    ('table', 'bàn', '桌子'),
    ('coffee table', 'bàn trà', '咖啡桌'),
    ('cabinet', 'tủ', '橱柜'),
    ('wardrobe', 'tủ quần áo', '衣柜'),
    ('drawer', 'ngăn kéo', '抽屉'),
    ('bookcase', 'tủ sách', '书架'),
    ('curtain', 'rèm', '窗帘'),
    ('mirror', 'gương', '镜子'),
    ('painting', 'tranh vẽ', '绘画'),
    ('lamp', 'đèn', '灯'),
    ('air conditioner', 'máy điều hòa', '空调'),
    ('printer', 'máy in', '打印机'),
    ('speaker', 'loa', '扬声器'),
    ('earphone', 'tai nghe', '耳机'),
    ('radio receiver', 'đài radio', '收音机'),
    ('wall socket', 'ổ cắm điện', '墙上插座'),
    ('flashlight', 'đèn pin', '手电筒'),
    ('bulletin board', 'bảng thông báo', '公告板'),
    ('pen', 'bút mực', '钢笔'),
    ('pencil', 'bút chì', '铅笔'),
    ('marker', 'bút lông', '记号笔'),
    ('eraser', 'cục tẩy', '橡皮擦'),
    ('measuring stick', 'thước', '测量杆'),
    ('newspaper', 'giấy', '报纸'),
    ('envelope', 'phong bì', '信封'),
    ('magazine', 'tạp chí', '杂志'),
    ('calendar', 'lịch', '日历'),
    ('calculator', 'máy tính cầm tay', '计算器'),
    ('toy', 'đồ chơi', '玩具'),
    ('doll', 'búp bê', '玩偶'),
    ('balloon', 'bóng bay', '气球'),
    ('sleeping bag', 'túi ngủ', '睡袋'),
    ('lantern', 'đèn lồng', '灯笼'),
    ('candle', 'nến', '蜡烛'),
    ('matchbox', 'hộp diêm', '火柴盒'),
    ('fire extinguisher', 'bình chữa cháy', '灭火器'),
    ('toolbox', 'hộp dụng cụ', '工具箱'),
    ('hammer', 'búa', '锤子'),
    ('screwdriver', 'tua vít', '螺丝刀'),
    ('wrench', 'cờ lê', '扳手'),
    ('drill', 'máy khoan', '钻头'),
    ('shovel', 'xẻng', '铲子'),
    ('ax', 'rìu', '斧子'),
    ('ladder', 'thang', '梯子'),
    ('bucket', 'xô', '桶'),
    ('broom', 'chổi', '扫帚'),
    ('mop', 'cây lau nhà', '拖把'),
    ('trash can', 'thùng rác', '垃圾桶'),
    ('vacuum cleaner', 'máy hút bụi', '吸尘器'),
    ('sewing machine', 'máy may', '缝纫机'),
    ('iron', 'bàn ủi', '熨斗'),
    ('automatic washer', 'máy giặt', '洗衣机'),
    ('soccer ball', 'bóng đá', '足球'),
    ('basketball', 'bóng rổ', '篮球'),
    ('volleyball', 'bóng chuyền', '排球'),
    ('golf club', 'gậy golf', '高尔夫球杆'),
    ('boxing glove', 'găng đấm bốc', '拳击手套'),
    ('dumbbell', 'tạ tay', '哑铃'),
    ('barbell', 'tạ đòn', '杠铃'),
    ('ski', 'ván trượt tuyết', '滑雪板'),
    ('guitar', 'đàn ghi-ta', '吉他'),
    ('piano', 'đàn piano', '钢琴'),
    ('violin', 'đàn vĩ cầm', '小提琴'),
    ('cornet', 'kèn trumpet', '短号'),
    ('saxophone', 'kèn saxophone', '萨克斯管'),
    ('musical instrument', 'nhạc cụ', '乐器'),
    ('scoreboard', 'bảng điểm', '记分牌'),
    ('crown', 'vương miện', '王冠'),
    ('banner', 'băng rôn', '横幅'),
    ('syringe', 'ống tiêm', '注射器'),
    ('medicine', 'thuốc', '药'),
    ('crutch', 'nạng', '拐杖'),
    ('award', 'giải thưởng', '奖品'),
    ('bamboo', 'tre', '竹子'),
    ('ball', 'quả bóng', '球'),
    ('bell', 'chuông', '铃铛'),
    ('binoculars', 'ống nhòm', '双筒望远镜'),
    ('blanket', 'chăn', '毛毯'),
    ('camcorder', 'máy quay cầm tay', '便携式摄像机'),
    ('carton', 'thùng giấy', '纸板箱'),
    ('cash register', 'máy tính tiền', '收银机'),
    ('chime', 'cồng', '钟声'),
]
if len(LABEL_SPECS) != 400:
    raise RuntimeError(f"Core vocabulary must contain 400 classes, got {len(LABEL_SPECS)}")
LABELS_EN = [row[0] for row in LABEL_SPECS]
LABELS_VI = [row[1] for row in LABEL_SPECS]
LABELS_ZH = [row[2] for row in LABEL_SPECS]
if len(LABELS_ZH) != len(set(LABELS_ZH)):
    raise RuntimeError("Duplicate Chinese prompt; class ids would be ambiguous")

VOCABULARY_HASH = hashlib.sha256(
    json.dumps(LABEL_SPECS, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
).hexdigest()

# Clone code chính thức và dùng đường inference thuần PyTorch. Một patch nhỏ
# đặt weights_only=False vì checkpoint chính thức chứa metadata MMEngine và
# PyTorch mới đổi mặc định torch.load; chỉ áp dụng cho checkpoint đã cố định hash.
if SOURCE_ROOT.exists() and not (SOURCE_ROOT / ".git").exists():
    shutil.rmtree(SOURCE_ROOT)
if not (SOURCE_ROOT / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/WeChatCV/WeDetect.git", str(SOURCE_ROOT)],
        check=True,
    )
WEDETECT_COMMIT = subprocess.check_output(
    ["git", "-C", str(SOURCE_ROOT), "rev-parse", "HEAD"], text=True
).strip()

standalone_path = SOURCE_ROOT / "deploy" / "test_coco_pytorch.py"
standalone_source = standalone_path.read_text(encoding="utf-8")
original_standalone_source = standalone_source
standalone_source = standalone_source.replace(
    "torch.load(ckpt_path, map_location=\"cpu\")",
    "torch.load(ckpt_path, map_location=\"cpu\", weights_only=False)",
).replace(
    "torch.load(model_path, map_location='cpu')",
    "torch.load(model_path, map_location='cpu', weights_only=False)",
)
if standalone_source == original_standalone_source and "weights_only=False" not in standalone_source:
    raise RuntimeError("Official WeDetect loader changed; review torch.load before loading checkpoint")
standalone_path.write_text(standalone_source, encoding="utf-8")

checkpoint_path = Path(hf_hub_download(
    repo_id=MODEL_REPO, filename=MODEL_FILENAME, token=HF_TOKEN,
    local_dir=ROOT / "model",
))
with checkpoint_path.open("rb") as stream:
    digest = hashlib.sha256()
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        digest.update(block)
MODEL_SHA256 = digest.hexdigest()

spec = importlib.util.spec_from_file_location("wedetect_standalone", standalone_path)
wedetect = importlib.util.module_from_spec(spec)
spec.loader.exec_module(wedetect)

# Encode vocabulary đúng một lần; giải phóng text tower trước khi tạo hai vision tower.
text_device = torch.device(f"cuda:{GPU_IDS[0]}")
language_encoder = wedetect.XLMRobertaLanguageBackbone(
    "FacebookAI/xlm-roberta-large", str(checkpoint_path)
).to(text_device).eval()
with torch.inference_mode():
    text_embeddings_cpu = torch.nn.functional.normalize(
        language_encoder(LABELS_ZH), dim=-1
    ).float().cpu().squeeze()
del language_encoder
gc.collect()
torch.cuda.empty_cache()

models = []
text_embeddings = []
for device_id in GPU_IDS:
    detector = wedetect.SimpleYOLOWorldDetector(
        MODEL_VARIANT,
        score_thr=CONFIDENCE_THRESHOLD,
        nms_iou=IOU_THRESHOLD,
        post_nms_topk=MAX_DETECTIONS,
    )
    wedetect.load_vision_checkpoint(detector, str(checkpoint_path))
    detector = detector.to(f"cuda:{device_id}").eval()
    models.append(detector)
    text_embeddings.append(text_embeddings_cpu.to(f"cuda:{device_id}"))
    print(f"Loaded WeDetect-{MODEL_VARIANT} on cuda:{device_id}")

print("WeDetect commit:", WEDETECT_COMMIT)
print("Checkpoint SHA256:", MODEL_SHA256)
print("Vocabulary classes:", len(LABEL_SPECS), VOCABULARY_HASH)

# Warmup bằng JPEG thật vì implementation chính thức nhận danh sách đường dẫn.
warmup_path = ROOT / "warmup.jpg"
Image.new("RGB", (640, 480), "black").save(warmup_path)
for detector, embeddings, device_id in zip(models, text_embeddings, GPU_IDS):
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        detector([str(warmup_path)], embeddings)
    torch.cuda.synchronize(device_id)
    print(f"Warmup OK on cuda:{device_id}")



In [ ]:
DOWNLOAD_ATTEMPTS = 5
DOWNLOAD_IDLE_SECONDS = 120
DOWNLOAD_ATTEMPT_SECONDS = 1200


def directory_bytes(directory):
    total = 0
    for path in directory.rglob("*"):
        try:
            if path.is_file():
                total += path.stat().st_size
        except FileNotFoundError:
            pass
    return total


def download_hf_file(filename, local_dir):
    """Bounded HF download; a stalled transfer is terminated and resumed."""
    local_dir.mkdir(parents=True, exist_ok=True)
    destination = local_dir / Path(filename)
    child_source = r'''
import os
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id=os.environ["OD_INPUT_REPO"],
    repo_type="dataset",
    revision=os.environ["OD_INPUT_REVISION"],
    filename=os.environ["OD_FILENAME"],
    token=os.environ["HF_TOKEN"],
    local_dir=os.environ["OD_LOCAL_DIR"],
)
'''
    child_env = os.environ.copy()
    child_env.update({
        "HF_TOKEN": HF_TOKEN,
        "OD_INPUT_REPO": INPUT_REPO,
        "OD_INPUT_REVISION": INPUT_REVISION,
        "OD_FILENAME": filename,
        "OD_LOCAL_DIR": str(local_dir),
        "HF_HUB_DISABLE_XET": "1",
        "HF_HUB_ENABLE_HF_TRANSFER": "0",
        "HF_HUB_DOWNLOAD_TIMEOUT": "60",
        "HF_HUB_ETAG_TIMEOUT": "30",
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
        "PYTHONUNBUFFERED": "1",
    })
    log_path = local_dir / "download.log"

    for attempt in range(1, DOWNLOAD_ATTEMPTS + 1):
        previous = directory_bytes(local_dir)
        started = last_change = time.monotonic()
        reason = "download failed"
        with log_path.open("w", encoding="utf-8") as log:
            process = subprocess.Popen(
                [sys.executable, "-u", "-c", child_source],
                env=child_env, stdout=log, stderr=subprocess.STDOUT,
            )
            try:
                while True:
                    try:
                        return_code = process.wait(timeout=10)
                        break
                    except subprocess.TimeoutExpired:
                        now = time.monotonic()
                        current = directory_bytes(local_dir)
                        if current != previous:
                            last_change = now
                        previous = current
                        if now - last_change >= DOWNLOAD_IDLE_SECONDS:
                            reason = "no progress for 120 seconds"
                            return_code = None
                            break
                        if now - started >= DOWNLOAD_ATTEMPT_SECONDS:
                            reason = "attempt exceeded 20 minutes"
                            return_code = None
                            break
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()

        if return_code == 0 and destination.is_file():
            return destination
        if return_code is not None:
            reason = f"child process exited {return_code}"
        print(f"Retry {filename}: {reason} [{attempt}/{DOWNLOAD_ATTEMPTS}]")
        if attempt < DOWNLOAD_ATTEMPTS:
            time.sleep(min(60, 10 * attempt))

    raise RuntimeError(
        f"Could not download {filename}; diagnostic log: {log_path}"
    )


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def locate_tar_member(archive, image_path):
    expected = PurePosixPath(str(image_path).replace("\\", "/")).as_posix()
    try:
        return archive.getmember(expected)
    except KeyError:
        matches = [
            member for member in archive.getmembers()
            if member.isfile() and PurePosixPath(member.name).name == PurePosixPath(expected).name
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"Cannot uniquely locate {image_path!r} inside keyframes.tar"
            )
        return matches[0]


def materialize_batch(archive, frame_records):
    """Write selected tar members to a short-lived directory for WeDetect."""
    image_paths = []
    for record in frame_records:
        member = locate_tar_member(archive, record["image_path"])
        stream = archive.extractfile(member)
        if stream is None:
            raise RuntimeError(f"Cannot read tar member {member.name}")
        suffix = PurePosixPath(member.name).suffix.lower()
        if suffix not in {".jpg", ".jpeg", ".png", ".webp"}:
            suffix = ".jpg"
        image_dir = DECODE_ROOT / safe_name(record["video_id"])
        image_dir.mkdir(parents=True, exist_ok=True)
        image_path = image_dir / f"{int(record['sample_n']):06d}{suffix}"
        with image_path.open("wb") as output:
            shutil.copyfileobj(stream, output)
        with Image.open(image_path) as image:
            record["_image_width"], record["_image_height"] = map(int, image.size)
        image_paths.append(image_path)
    return image_paths


def infer_adaptive(detector, embeddings, device_id, archive, records, batch_size):
    """Yield aligned (metadata, result), reducing batch after CUDA OOM."""
    cursor = 0
    active_batch = batch_size
    while cursor < len(records):
        selected = records[cursor:cursor + active_batch]
        image_paths = materialize_batch(archive, selected)
        try:
            with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
                results = detector([str(path) for path in image_paths], embeddings)
            if len(results) != len(selected):
                raise RuntimeError("Detector returned a mismatched batch length")
        except (torch.OutOfMemoryError, RuntimeError) as error:
            is_oom = isinstance(error, torch.OutOfMemoryError) or "out of memory" in str(error).lower()
            if not is_oom or active_batch == 1:
                raise
            active_batch = max(1, active_batch // 2)
            print(f"cuda:{device_id} OOM; reducing batch to {active_batch}")
            for path in image_paths:
                path.unlink(missing_ok=True)
            gc.collect()
            with torch.cuda.device(device_id):
                torch.cuda.empty_cache()
            continue

        # Move results to CPU before yielding so each worker holds little VRAM.
        cpu_results = [
            {
                key: value.detach().float().cpu() if torch.is_tensor(value) else value
                for key, value in result.items()
            }
            for result in results
        ]
        del results
        try:
            for record, result, image_path in zip(selected, cpu_results, image_paths):
                yield record, result, image_path, active_batch
        finally:
            for path in image_paths:
                path.unlink(missing_ok=True)
        cursor += len(selected)
        del cpu_results


DETECTION_COLUMNS = [
    "video_id", "frame_uid", "sample_n", "frame_idx", "shot_id",
    "timestamp_sec", "image_path", "class_id", "label_en", "label_vi", "label_zh",
    "confidence", "x1", "y1", "x2", "y2", "x1_norm", "y1_norm",
    "x2_norm", "y2_norm", "center_x_norm", "center_y_norm",
    "position_horizontal", "position_vertical", "box_area_ratio",
    "image_width", "image_height",
    "model_id", "model_sha256", "vocabulary_hash", "core_vocab_version",
]
FRAME_COLUMNS = [
    "video_id", "frame_uid", "sample_n", "frame_idx", "shot_id",
    "timestamp_sec", "image_path", "image_width", "image_height",
    "detection_count", "max_confidence", "status",
]


def result_rows(record, result):
    width = int(record["_image_width"])
    height = int(record["_image_height"])
    frame_uid = f"{record['video_id']}:{int(record['frame_idx'])}"
    detections = []
    xyxy = result["bboxes"].numpy()
    scores = result["scores"].numpy()
    classes = result["labels"].long().numpy()
    if len(xyxy):
        for coordinates, score, class_id in zip(xyxy, scores, classes):
            x1, y1, x2, y2 = map(float, coordinates)
            class_id = int(class_id)
            if not 0 <= class_id < len(LABEL_SPECS):
                raise RuntimeError(f"WeDetect returned invalid class id {class_id}")
            label_en, label_vi, label_zh = LABEL_SPECS[class_id]
            center_x_norm = ((x1 + x2) / 2) / width
            center_y_norm = ((y1 + y2) / 2) / height
            position_horizontal = (
                "left" if center_x_norm < 1 / 3 else
                "right" if center_x_norm > 2 / 3 else "center"
            )
            position_vertical = (
                "top" if center_y_norm < 1 / 3 else
                "bottom" if center_y_norm > 2 / 3 else "middle"
            )
            detections.append({
                "video_id": record["video_id"], "frame_uid": frame_uid,
                "sample_n": int(record["sample_n"]),
                "frame_idx": int(record["frame_idx"]),
                "shot_id": int(record["shot_id"]),
                "timestamp_sec": float(record["timestamp_sec"]),
                "image_path": str(record["image_path"]), "class_id": class_id,
                "label_en": label_en, "label_vi": label_vi, "label_zh": label_zh,
                "confidence": float(score), "x1": x1, "y1": y1,
                "x2": x2, "y2": y2, "x1_norm": x1 / width,
                "y1_norm": y1 / height, "x2_norm": x2 / width,
                "y2_norm": y2 / height,
                "center_x_norm": center_x_norm, "center_y_norm": center_y_norm,
                "position_horizontal": position_horizontal,
                "position_vertical": position_vertical,
                "box_area_ratio": max(0.0, x2 - x1) * max(0.0, y2 - y1) / (width * height),
                "image_width": width, "image_height": height,
                "model_id": MODEL_ID, "model_sha256": MODEL_SHA256,
                "vocabulary_hash": VOCABULARY_HASH,
                "core_vocab_version": CORE_VOCAB_VERSION,
            })
    frame = {
        "video_id": record["video_id"], "frame_uid": frame_uid,
        "sample_n": int(record["sample_n"]),
        "frame_idx": int(record["frame_idx"]),
        "shot_id": int(record["shot_id"]),
        "timestamp_sec": float(record["timestamp_sec"]),
        "image_path": str(record["image_path"]),
        "image_width": width, "image_height": height,
        "detection_count": len(detections),
        "max_confidence": max((row["confidence"] for row in detections), default=0.0),
        "status": "detected" if detections else "empty_valid",
    }
    return detections, frame


def save_preview(image_path, result, destination):
    """Save a lightweight QA overlay without an Ultralytics dependency."""
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    draw = ImageDraw.Draw(image)
    boxes = result["bboxes"].numpy()
    scores = result["scores"].numpy()
    classes = result["labels"].long().numpy()
    for coordinates, score, class_id in zip(boxes, scores, classes):
        x1, y1, x2, y2 = map(float, coordinates)
        label = f"{LABELS_EN[int(class_id)]} {float(score):.2f}"
        draw.rectangle((x1, y1, x2, y2), outline="red", width=3)
        left, top, right, bottom = draw.textbbox((x1, y1), label)
        draw.rectangle((left, top, right + 4, bottom + 4), fill="red")
        draw.text((x1 + 2, y1 + 2), label, fill="white")
    image.save(destination, quality=90)


def write_video_output(video_id, detection_rows, frame_rows, stats):
    level = video_id.split("_")[0]
    directory = OUTPUT_ROOT / level / video_id
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)
    detections_path = directory / "detections.parquet"
    frames_path = directory / "frames.parquet"
    marker_path = directory / "_OD_SUCCESS.json"
    pd.DataFrame(detection_rows, columns=DETECTION_COLUMNS).to_parquet(
        detections_path, index=False
    )
    pd.DataFrame(frame_rows, columns=FRAME_COLUMNS).to_parquet(frames_path, index=False)
    marker = {
        "video_id": video_id,
        "status": "success",
        "frames": len(frame_rows),
        "frames_with_detections": sum(row["detection_count"] > 0 for row in frame_rows),
        "detections": len(detection_rows),
        "source_repo": INPUT_REPO,
        "source_revision": INPUT_REVISION,
        "model_id": MODEL_ID,
        "model_repo": MODEL_REPO,
        "model_variant": MODEL_VARIANT,
        "model_sha256": MODEL_SHA256,
        "wedetect_commit": WEDETECT_COMMIT,
        "vocabulary_hash": VOCABULARY_HASH,
        "core_vocab_version": CORE_VOCAB_VERSION,
        "prompt_language": "zh",
        "vocabulary": [
            {"label_en": en, "label_vi": vi, "label_zh": zh}
            for en, vi, zh in LABEL_SPECS
        ],
        "image_size": IMAGE_SIZE,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "precision": "fp16",
        "stats": stats,
        "schema_version": 2,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
    return [detections_path, frames_path, marker_path]


def upload_outputs(paths, message):
    operations = [
        CommitOperationAdd(
            path_in_repo=f"data/{path.relative_to(OUTPUT_ROOT).as_posix()}",
            path_or_fileobj=str(path),
        )
        for path in paths
    ]
    for attempt in range(1, 8):
        try:
            api.create_commit(
                repo_id=OUTPUT_REPO, repo_type="dataset",
                operations=operations, commit_message=message, token=HF_TOKEN,
            )
            return
        except Exception as error:
            if attempt == 7:
                raise
            delay = min(300, 10 * (2 ** (attempt - 1)))
            print(f"Upload error {type(error).__name__}; retry in {delay}s")
            time.sleep(delay)



In [ ]:
input_files = set(api.list_repo_files(INPUT_REPO, repo_type="dataset", token=HF_TOKEN))
output_files = set(api.list_repo_files(OUTPUT_REPO, repo_type="dataset", token=HF_TOKEN))

tar_pattern = re.compile(r"^data/(L(?:2[1-9]|30))/(L\d+_V\d+)/keyframes\.tar$")
video_specs = []
for filename in sorted(input_files):
    match = tar_pattern.fullmatch(filename)
    if not match:
        continue
    level, video_id = match.groups()
    frames_filename = f"data/{level}/{video_id}/frames.parquet"
    if frames_filename not in input_files:
        raise RuntimeError(f"Missing source metadata: {frames_filename}")
    video_specs.append({
        "level": level,
        "video_id": video_id,
        "tar_filename": filename,
        "frames_filename": frames_filename,
    })


def remote_complete(spec):
    prefix = f"data/{spec['level']}/{spec['video_id']}"
    return all(
        f"{prefix}/{name}" in output_files
        for name in ("detections.parquet", "frames.parquet", "_OD_SUCCESS.json")
    )


completed_specs = [spec for spec in video_specs if remote_complete(spec)]
pending_specs = [spec for spec in video_specs if not remote_complete(spec)]
if MAX_VIDEOS_PER_RUN is not None:
    pending_specs = pending_specs[:MAX_VIDEOS_PER_RUN]

QA_VIDEO_IDS = {spec["video_id"] for spec in pending_specs[:QA_PREVIEW_COUNT]}
print("Source videos:", len(video_specs))
print("Already complete:", len(completed_specs))
print("Selected this run:", len(pending_specs))

device_pool = queue.Queue()
for model_index in range(len(models)):
    device_pool.put(model_index)


def process_video(spec):
    model_index = device_pool.get()
    detector = models[model_index]
    embeddings = text_embeddings[model_index]
    device_id = GPU_IDS[model_index]
    video_id = spec["video_id"]
    task_dir = DOWNLOAD_ROOT / safe_name(video_id)
    shutil.rmtree(task_dir, ignore_errors=True)
    task_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    try:
        frames_path = download_hf_file(spec["frames_filename"], task_dir)
        tar_path = download_hf_file(spec["tar_filename"], task_dir)
        metadata = pd.read_parquet(frames_path).sort_values("sample_n")
        required = {"video_id", "sample_n", "shot_id", "frame_idx", "timestamp_sec", "image_path"}
        missing = required - set(metadata.columns)
        if missing:
            raise RuntimeError(f"{video_id}: source columns missing: {sorted(missing)}")
        if metadata.empty:
            raise RuntimeError(f"{video_id}: source keyframe metadata is empty")
        if metadata["video_id"].astype(str).nunique() != 1 or str(metadata.iloc[0]["video_id"]) != video_id:
            raise RuntimeError(f"{video_id}: inconsistent source video_id")

        records = metadata[list(required)].to_dict("records")
        detection_rows = []
        frame_rows = []
        preview_saved = False
        minimum_batch = INITIAL_BATCH_SIZE
        torch.cuda.reset_peak_memory_stats(device_id)
        with tarfile.open(tar_path, mode="r:*") as archive:
            for record, result, image_path, active_batch in infer_adaptive(
                detector, embeddings, device_id, archive, records, INITIAL_BATCH_SIZE
            ):
                rows, frame = result_rows(record, result)
                detection_rows.extend(rows)
                frame_rows.append(frame)
                minimum_batch = min(minimum_batch, active_batch)
                if len(frame_rows) % 100 == 0 or len(frame_rows) == len(records):
                    print(
                        f"  cuda:{device_id} {video_id}: "
                        f"{len(frame_rows)}/{len(records)} frames"
                    )
                should_preview = bool(rows) or len(frame_rows) == len(records)
                if video_id in QA_VIDEO_IDS and not preview_saved and should_preview:
                    save_preview(image_path, result, PREVIEW_ROOT / f"{video_id}.jpg")
                    preview_saved = True

        if len(frame_rows) != len(records):
            raise RuntimeError(
                f"{video_id}: processed {len(frame_rows)}/{len(records)} frames"
            )
        torch.cuda.synchronize(device_id)
        elapsed = time.perf_counter() - started
        stats = {
            "elapsed_sec": round(elapsed, 3),
            "frames_per_sec": round(len(frame_rows) / max(elapsed, 1e-6), 3),
            "peak_vram_gib": round(torch.cuda.max_memory_allocated(device_id) / 1024**3, 3),
            "minimum_batch_size": minimum_batch,
            "device": f"cuda:{device_id}",
        }
        paths = write_video_output(video_id, detection_rows, frame_rows, stats)
        return {
            "video_id": video_id, "paths": paths, "stats": stats,
            "frames": len(frame_rows), "detections": len(detection_rows),
        }
    finally:
        shutil.rmtree(task_dir, ignore_errors=True)
        shutil.rmtree(DECODE_ROOT / safe_name(video_id), ignore_errors=True)
        gc.collect()
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
        device_pool.put(model_index)


pending_paths = []
pending_ids = []
run_results = []
failures = []


def flush_pending():
    global pending_paths, pending_ids
    if not pending_paths:
        return
    upload_outputs(
        pending_paths,
        f"Add WeDetect-{MODEL_VARIANT} OD for {pending_ids[0]} through {pending_ids[-1]}",
    )
    print(f"✅ Uploaded {len(pending_ids)} videos in one commit")
    for video_id in pending_ids:
        shutil.rmtree(OUTPUT_ROOT / video_id.split("_")[0] / video_id, ignore_errors=True)
    pending_paths = []
    pending_ids = []


with ThreadPoolExecutor(max_workers=len(models)) as executor:
    futures = {executor.submit(process_video, spec): spec for spec in pending_specs}
    for position, future in enumerate(as_completed(futures), 1):
        spec = futures[future]
        try:
            result = future.result()
            run_results.append(result)
            pending_paths.extend(result["paths"])
            pending_ids.append(result["video_id"])
            stats = result["stats"]
            print(
                f"[{position}/{len(futures)}] ✅ {result['video_id']}: "
                f"{result['frames']} frames, {result['detections']} detections, "
                f"{stats['frames_per_sec']} FPS, {stats['peak_vram_gib']} GiB"
            )
            if len(pending_ids) >= UPLOAD_BATCH_VIDEOS:
                flush_pending()
        except Exception as error:
            failures.append({"video_id": spec["video_id"], "error": repr(error)})
            print(f"[{position}/{len(futures)}] ❌ {spec['video_id']}: {error!r}")

flush_pending()

print("=" * 72)
print("Processed this run:", len(run_results))
print("Failures:", len(failures))
if run_results:
    total_frames = sum(item["frames"] for item in run_results)
    total_seconds = sum(item["stats"]["elapsed_sec"] for item in run_results)
    print("Frames:", total_frames)
    print("Aggregate worker FPS:", round(total_frames / max(total_seconds, 1e-6), 3))
if failures:
    print(json.dumps(failures, ensure_ascii=False, indent=2))
    raise RuntimeError("OD run has failures. Rerun; completed remote videos will be skipped.")



In [ ]:
# Remote completion audit. Refresh the listing after all commits.
final_files = set(api.list_repo_files(OUTPUT_REPO, repo_type="dataset", token=HF_TOKEN))
missing_videos = []
for spec in video_specs:
    prefix = f"data/{spec['level']}/{spec['video_id']}"
    if not all(
        f"{prefix}/{name}" in final_files
        for name in ("detections.parquet", "frames.parquet", "_OD_SUCCESS.json")
    ):
        missing_videos.append(spec["video_id"])

if MAX_VIDEOS_PER_RUN is None and missing_videos:
    raise RuntimeError(f"Missing OD outputs for {len(missing_videos)} videos: {missing_videos[:30]}")

print(f"✅ Remote complete: {len(video_specs) - len(missing_videos)}/{len(video_specs)} videos")
if missing_videos:
    print("Remaining for next run:", len(missing_videos))
print("Dataset:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")



In [ ]:
# Visual QA: ảnh đã vẽ box của một số video đầu tiên trong phiên này.
from IPython.display import display

preview_paths = sorted(PREVIEW_ROOT.glob("*.jpg"))
print("Preview images:", len(preview_paths))
for path in preview_paths[:QA_PREVIEW_COUNT]:
    print(path.name)
    display(Image.open(path))


